## Build two tower model

In [1]:
!pwd

/home/jupyter/crispy_towers/notebooks


In [2]:
import sys
sys.path.append("..")
import env_config

In [3]:
print(f"PREFIX: {env_config.PREFIX}")
print(f"PROJECT_ID: {env_config.PROJECT_ID}")
print(f"LOCATION: {env_config.LOCATION}")

PREFIX: jt-towers-v1
PROJECT_ID: hybrid-vertex
LOCATION: us-central1


## imports

In [5]:
import os
import json
from pprint import pprint
import pickle as pkl
import numpy as np
import pandas as pd

import logging
logging.disable(logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

# tensorflow
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
# os.environ['TF_USE_LEGACY_KERAS'] = '1'
import tensorflow as tf
import tensorflow_datasets as tfds
# import tensorflow_recommenders as tfrs

# google cloud
from google.cloud import aiplatform, storage

# cloud storage client
storage_client = storage.Client(project=env_config.PROJECT_ID)
# bucket = storage_client.bucket(BUCKET_NAME)

# Vertex client
aiplatform.init(project=env_config.PROJECT_ID, location=env_config.LOCATION)


print(f"Vertex AI SDK version = {aiplatform.__version__}")
print(f"Tensorflow version = {tf.__version__}")
# print(f"Tensorflow Recommenders (tfrs) version = {tfrs.__version__}")

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [6]:
sys.path.append("..")
from src.data import data_utils as data_utils
from src.model import two_tower

### detect GPU

In [8]:
# GPU
import gc
from numba import cuda

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  0


In [10]:
device = cuda.get_current_device()
device.reset()
gc.collect()

14

## Get data

In [11]:
GCS_DATA_PATH = f"{env_config.BUCKET_URI}/{env_config.EXAMPLE_GEN_GCS_PATH}"
print(f"GCS_DATA_PATH : {GCS_DATA_PATH}")

! gsutil ls $GCS_DATA_PATH

GCS_DATA_PATH : gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/val/
gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/vocabs/


### train records

In [12]:
train_files = []

for blob in storage_client.list_blobs(
    f"{env_config.BUCKET_NAME}", 
    prefix=f'{env_config.EXAMPLE_GEN_GCS_PATH}/train/', 
    # delimiter='/'
):
    if '.tfrecord' in blob.name:
        train_files.append(blob.public_url.replace("https://storage.googleapis.com/", "gs://"))
        
train_files

['gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-001-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-002-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-003-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-004-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-005-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-006-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-007-of-008.tfrecord',
 'gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/train/ml1m-008-of-008.tfrecord']

In [13]:
mv_dataset = tf.data.TFRecordDataset(train_files)
# train_dataset = train_dataset.map(movielens_ds_utils.parse_tfrecord)
mv_dataset = mv_dataset.map(data_utils._parse_function)

for x in mv_dataset.batch(1).take(1):
    pprint(x)

{'context_movie_genre': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
array([[b'Animation', b'Comedy', b'Thriller', b'Animation',
        b"Children's", b'Comedy', b'Musical', b'Animation',
        b"Children's", b'Comedy']], dtype=object)>,
 'context_movie_id': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
array([[b'735', b'2009', b'2012', b'2965', b'3682', b'907', b'1268',
        b'2231', b'887', b'902']], dtype=object)>,
 'context_movie_rating': <tf.Tensor: shape=(1, 10), dtype=float32, numpy=array([[5., 4., 3., 2., 4., 3., 5., 4., 3., 4.]], dtype=float32)>,
 'context_movie_title': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
array([[b'Close Shave, A (1995)', b'Jungle Book, The (1967)',
        b'Little Mermaid, The (1989)', b'Robin Hood (1973)',
        b'Chicken Run (2000)', b'Wizard of Oz, The (1939)',
        b'This Is Spinal Tap (1984)', b'Producers, The (1968)',
        b"Singin' in the Rain (1952)", b'My Fair Lady (1964)']],
      dtype=object)>,
 'context_movie_year'

### Candidate dataset

In [14]:
# candidate_features = {
#     'target_movie_id': tf.io.FixedLenFeature(shape=(), dtype=tf.string),
#     'target_movie_genre': tf.io.FixedLenFeature(shape=(), dtype=tf.string),
#     'target_movie_year': tf.io.FixedLenFeature(shape=(), dtype=tf.int64),
#     'target_movie_title': tf.io.FixedLenFeature(shape=(), dtype=tf.string),
# }
# def _parse_candidates_fn(example_proto):
#     return tf.io.parse_single_example(
#         example_proto, candidate_features
#     )

CANDIDATE_FILES = [
    "gs://jt-towers-v1-hybrid-vertex-bucket/candidates/train/output-00000-of-00001.tfrecord"
]

In [15]:
candidate_dataset = tf.data.TFRecordDataset(CANDIDATE_FILES)

parsed_candidate_dataset = candidate_dataset.map(data_utils._parse_candidates_fn)
# parsed_candidate_dataset = parsed_candidate_dataset.cache()

In [16]:
for x in parsed_candidate_dataset.batch(1).take(1):
    pprint(x)

{'target_movie_genre': <tf.Tensor: shape=(1,), dtype=string, numpy=array([b'Action'], dtype=object)>,
 'target_movie_id': <tf.Tensor: shape=(1,), dtype=string, numpy=array([b'1480'], dtype=object)>,
 'target_movie_title': <tf.Tensor: shape=(1,), dtype=string, numpy=array([b"Smilla's Sense of Snow (1997)"], dtype=object)>,
 'target_movie_year': <tf.Tensor: shape=(1,), dtype=int64, numpy=array([1997])>}


### vocab file

In [17]:
VOCAB_FILENAME="vocab_dict.pkl"

In [18]:
EXISTING_VOCAB_FILE = f'gs://{env_config.BUCKET_NAME}/{env_config.EXAMPLE_GEN_GCS_PATH}/vocabs/{VOCAB_FILENAME}'
print(f"Downloading vocab...")

os.system(f'gsutil -q cp {EXISTING_VOCAB_FILE} .')
print(f"Downloaded vocab from: {EXISTING_VOCAB_FILE}\n")

filehandler = open(VOCAB_FILENAME, 'rb')
vocab_dict = pkl.load(filehandler)
filehandler.close()

for key in vocab_dict.keys():
    pprint(key)

Downloaded vocab from: gs://jt-towers-v1-hybrid-vertex-bucket/data/movielens/m1m/vocabs/vocab_dict.pkl

'movie_id'
'movie_year'
'movie_genre'
'movie_title'
'user_id'
'user_gender_vocab'
'user_age_vocab'
'user_occ_vocab'
'user_zip_vocab'
'min_timestamp'
'max_timestamp'
'timestamp_buckets'


# Build and compile two-tower model

In [19]:
USE_CROSS_LAYER = True
USE_DROPOUT = True
SEED = 1234
EMBEDDING_DIM = 128   
PROJECTION_DIM = int(EMBEDDING_DIM / 4) # 50  
SEED = 1234
DROPOUT_RATE = 0.33
MAX_TOKENS = 20000
LAYER_SIZES=[256,128]

LR = .1
opt = tf.keras.optimizers.Adagrad(LR)

print(f"PROJECTION_DIM: {PROJECTION_DIM}")

PROJECTION_DIM: 32


In [20]:
model = two_tower.TheTwoTowers(
    layer_sizes=LAYER_SIZES, 
    vocab_dict=vocab_dict, 
    parsed_candidate_dataset=parsed_candidate_dataset,
    embedding_dim=EMBEDDING_DIM,
    projection_dim=PROJECTION_DIM,
    seed=SEED,
    use_cross_layer=USE_CROSS_LAYER,
    use_dropout=USE_DROPOUT,
    dropout_rate=DROPOUT_RATE,
    max_tokens=MAX_TOKENS,
    max_context_length=env_config.MAX_CONTEXT_LENGTH,
    max_genre_length=env_config.MAX_GENRE_LENGTH,
    compute_batch_metrics=False
)

ValueError: Cannot convert '('c', 'o', 'u', 'n', 't', 'e', 'r')' to a shape. Found invalid entry 'c' of type '<class 'str'>'. 

In [ ]:
# pip install tensorflow==2.11.0 --force-reinstall --user

# pip install tensorflow-recommenders==0.7.2 --no-deps